In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import UIOrthoLoRAConfig, TaskType, get_peft_model

In [ ]:
def prepare_dataset(tokenizer, max_len=128, task="sst2"):
    ds = load_dataset("glue", task)
    
    def tokenize_function(examples):
        if task in ["sst2", "cola"]:
            # Single sentence tasks
            return tokenizer(
                examples["sentence"],
                truncation=True,
                padding="max_length",
                max_length=max_len
            )
        elif task in ["mrpc", "qnli", "rte", "wnli", "mnli", "qqp", "sts-b"]:
            # Two sentence tasks
            return tokenizer(
                examples["sentence1"],
                examples["sentence2"],
                truncation=True,
                padding="max_length",
                max_length=max_len
            )
    
    ds = ds.map(tokenize_function, batched=True)
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [3]:
base_id = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(base_id, use_fast=True)

base = AutoModelForSequenceClassification.from_pretrained(
    base_id, num_labels=2, device_map="auto"
)

/opt/anaconda3/envs/guyb_env2/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
uiortholora_cfg = UIOrthoLoRAConfig(
    target_modules=["query", "value"],
    uiortholora_alpha=1.0,
    uiortholora_dropout=0.0,
    fan_in_fan_out=False,
    initial_scaler=0.1,
    initial_sigma=0.1,
    num_svalues_to_adapt=128,
    num_svectors_to_adapt=60,
    task_type=TaskType.SEQ_CLS)
model = get_peft_model(base, uiortholora_cfg)

In [5]:
model.print_trainable_parameters()

trainable params: 804,866 || all params: 125,452,036 || trainable%: 0.6416


In [12]:
model.base_model.model.roberta.encoder.layer[2].attention.self.query.get_delta_weight("default")

U.shape:  torch.Size([768, 768])
Vt.shape:  torch.Size([768, 768])
right_unitary.shape:  torch.Size([768, 768])
VtD.shape:  torch.Size([768, 768])
EQ.shape:  torch.Size([768, 768])


tensor([[-4.0770e-04, -2.9242e-04,  2.5354e-04,  ..., -1.4987e-04,
          3.6572e-04, -3.7647e-04],
        [ 8.4685e-05,  1.5985e-04,  3.9045e-04,  ..., -1.8743e-06,
         -3.9857e-05, -4.3903e-04],
        [ 1.4674e-05,  6.7302e-05, -1.3020e-04,  ..., -1.2437e-04,
         -1.2958e-04, -1.2361e-04],
        ...,
        [ 7.6128e-05, -1.1162e-04,  2.3903e-04,  ..., -1.6908e-04,
          5.9994e-05,  3.1563e-04],
        [-5.7704e-05, -1.5014e-04, -1.9139e-04,  ..., -5.4565e-04,
         -1.0619e-05,  6.5060e-04],
        [ 1.5617e-04,  1.4803e-04,  3.8673e-04,  ...,  3.6590e-04,
          3.4393e-04,  2.0824e-04]], device='cuda:0', grad_fn=<MulBackward0>)

In [7]:
ortholora_layer = model.base_model.model.roberta.encoder.layer[0]

In [8]:
model.base_model.model.roberta.encoder.layer[0].attention.self.query.weight

Parameter containing:
tensor([[ 0.0729, -0.0029, -0.0902,  ...,  0.1033,  0.0900, -0.1030],
        [-0.0516,  0.2061,  0.0739,  ...,  0.0657,  0.0634,  0.1282],
        [ 0.0878,  0.0698, -0.0515,  ..., -0.0426, -0.0081,  0.1100],
        ...,
        [-0.1871,  0.0172, -0.0315,  ..., -0.0503,  0.1024, -0.1165],
        [-0.2532,  0.0439,  0.0638,  ...,  0.0701, -0.1045,  0.0118],
        [-0.0516, -0.0859,  0.1027,  ..., -0.1895,  0.0033, -0.0541]],
       device='cuda:0')

In [9]:
adapter_name = "default"
U = getattr(ortholora_layer, f"{adapter_name}_U")
V = getattr(ortholora_layer, f"{adapter_name}_V")

AttributeError: 'RobertaLayer' object has no attribute 'default_U'

In [15]:
dir(ortholora_layer)

['T_destination',
 '__abstractmethods__',
 '__annotations__',
 '__call__',
 '__class__',
 '__constants__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_active_adapter',
 '_active_adapter',
 '_all_available_adapter_names',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_build_projection_matrix',
 '_calc_left_unitary',
 '_calc_right_unitary',
 '_calc_sigma',
 '_call_impl',
 '_cast_input_dtype',
 '_compiled_call_impl',
 '_disable_adapters',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '

In [26]:
ortholora_layer.weight.shape

torch.Size([768, 768])

In [16]:
print(U.shape)
print(V.shape)

torch.Size([768, 128])
torch.Size([128, 768])
